# Congestion Rent: Pricing Europe's Grid Bottlenecks

When two neighbouring bidding zones have different day-ahead prices, power is supposed to flow from the cheap zone to the expensive one until prices converge. Sometimes it can't &ndash; the interconnector between them is already running at its physical capacity limit (its net transfer capacity, or NTC). 

This is called a **congestion** flow: the price spread persists because the grid, not the market, is the binding constraint.

Every hour a border sits congested, that persistent price spread has a measurable value attached to it: the price gap x volume traded is captured as **congestion rent** rather than competed away.

This notebook estimates that rent directly from flow data &ndash; for every congested observation, `|price difference| x |flow| x duration` gives the implied congestion rent in EUR &ndash; and then works through:

- a rolling 30-day leaderboard of Europe's highest-rent borders, plus the daily rent trend behind it
- a case study of 2025's most widely congested moment, ranked by how many borders were congested at once (a different question from which border is most expensive)
- monthly and yearly leaderboards across four competing metrics, since "worst border" depends on which one you ask
- a concentration check &ndash; do a handful of borders really dominate the total? &ndash; and a breakdown by region and country

Congestion rent is **not** the same thing as economic cost or lost welfare. See "Computing congestion rent" below for what this number is and isn't.

**Before you run this:** sign up at [voltpath.ai/dashboard](https://voltpath.ai/dashboard) for a free personal API key (copy it any time from [voltpath.ai/settings](https://voltpath.ai/settings)), then copy `.env.example` to `.env` and paste it in as `VOLTPATH_API_KEY`.

**Data source:** [ENTSO-E Transparency Platform](https://transparency.entsoe.eu), via the [Voltpath API](https://voltpath.ai) (docs at [docs.voltpath.ai](https://docs.voltpath.ai)). Physical flow data is CC-BY 4.0; day-ahead prices are reused under ENTSO-E's general Transparency Platform terms.

In [ ]:
import sys
from datetime import UTC, datetime, timedelta
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import plotly.express as px
from common import (
    FLOW_COLORS,
    REGION_COLORS,
    border_count_price_chart,
    border_country,
    border_region,
    congestion_rent_schematic,
    drop_ukraine,
    fetch_flows,
    fetch_flows_range,
    flow_type_summary,
    hourly_border_counts,
    leaderboard_subplots,
    save_chart,
    top_n_bar,
    top_vs_rest,
)
from plotly.subplots import make_subplots

## Pull flow data

`GET /api/v1/flows` already joins physical flow with day-ahead prices on both sides of the border and classifies each observation into `Convergence`, `Congestion`, `Counter-intuitive`, or `Outage` &ndash; see [Flow Types](https://docs.voltpath.ai/concepts/flow-types) for the full definitions.

`fetch_flows_range` (shared helper in [`articles/common.py`](../common.py)) fetches native-resolution data day by day, since the API caps how wide a single native-resolution request can be, and caches each day to disk so re-running this notebook doesn't re-hit the API for a range you've already pulled.

We will pull the last 30 days across every border at native resolution &ndash; see [Response Format: Resolution](https://docs.voltpath.ai/concepts/response-format#resolution) for what `native` means vs. the other options (`auto`, `1H`, `1D`, `1W`): `native` is raw, un-aggregated 15-minute data, capped to a 2-day range per request (hence the day-by-day chunking above); the others aggregate to coarser buckets to allow wider ranges in a single call.

In [ ]:
now = datetime.now(UTC)
flows_30d = fetch_flows_range(
    (now - timedelta(days=30)).strftime("%Y-%m-%d"), now.strftime("%Y-%m-%d")
)
print(f"{len(flows_30d):,} observations across {flows_30d['border'].nunique()} borders")
flows_30d.head()

## Computing congestion rent

Congestion rent per observation is `|price_delta| (EUR/MWh) x |net_flow_mw| (MW) x duration_h`, in EUR &ndash; `duration_h` and the signed version of this (`value_eur`) are already computed by `fetch_flows`.

This is a **congestion rent**: price gap x volume actually traded, the same "implied congestion income" ENTSO-E/ACER report under [Regulation (EU) 2019/943, Art. 19](https://eur-lex.europa.eu/eli/reg/2019/943/oj/eng) &ndash; see ENTSO-E's [TSOs' proposal for the Use of Congestion Income &ndash; explanatory document](https://eepublicdownloads.entsoe.eu/clean-documents/cep/200703_Use_of_Congestion_Income_-_Explanatory_Document.pdf), which defines congestion income as price differential x traded flow, the same formula used here.

Note that the congestion rent is a "rectangle" (observed price gap x observed flow), not the [deadweight-loss triangle](https://en.wikipedia.org/wiki/Deadweight_loss) you'd get from integrating under each zone's demand/supply curve &ndash; that would need bid-curve data. The schematic below shows both areas side by side.

Congestion rent is a defensible, directly observable severity signal: how much value the price gap is capturing while the border stays constrained, not a claim about how much better off Europe would be if the bottleneck were removed.

In [ ]:
fig = congestion_rent_schematic()
save_chart(fig, "congestion-rent-vs-dwl-schematic", scale=2)
fig.show()

With rent vs. deadweight loss now distinguished, the rest of this notebook computes the rent side only &ndash; the one we can actually observe from flow and price data. The cell below filters to `Congestion`-flagged observations and takes `|value_eur|` per observation as that observation's congestion rent.

In [ ]:
congested = flows_30d[flows_30d["flow_type"] == "Congestion"].copy()
congested["rent_eur"] = congested["value_eur"].abs()

total_rent = congested["rent_eur"].sum()
print(
    f"{len(congested):,} congested observations, congestion rent: EUR {total_rent:,.0f}"
)

## The leaderboard: highest-rent borders

A handful of interconnectors usually account for the bulk of the congestion rent. That's a claim worth testing rather than asserting, so the concentration check further down measures it directly.

Ukraine's `UA_IPS` borders (`HU/PL/RO/SK -> UA_IPS`) are a special case since 2022: wartime grid conditions and emergency EU synchronisation, not ordinary market congestion, drive their numbers, so they can dominate a leaderboard for reasons unrelated to everything else on it. The `exclude_ukraine` flag set below drops them, and applies to every leaderboard in this notebook &ndash; flip it once, here, to see them.

In [ ]:
exclude_ukraine = True  # UA_IPS: wartime special case, not ordinary congestion

leaderboard_30d = flow_type_summary(flows_30d, "Congestion")
if exclude_ukraine:
    leaderboard_30d = drop_ukraine(leaderboard_30d)

fig = top_n_bar(
    leaderboard_30d,
    "implied_value_eur",
    "Congestion rent (EUR)",
    "Top 15 borders by congestion rent (last 30 days)",
    FLOW_COLORS["Congestion"],
    top_n=15,
)
save_chart(fig, "top15-borders-congestion-rent-last-30-days")
fig.show()

## Congestion rent over time

Daily totals across all borders show whether rent is trending, seasonal, or driven by a handful of spike days.

In [ ]:
daily = congested.set_index("datetime")["rent_eur"].resample("1D").sum().reset_index()

fig = px.bar(
    daily,
    x="datetime",
    y="rent_eur",
    title="Daily congestion rent, all borders",
    labels={"datetime": "Day", "rent_eur": "Congestion rent (EUR)"},
    color_discrete_sequence=[FLOW_COLORS["Congestion"]],
)
save_chart(fig, "daily-congestion-rent-last-30-days")
fig.show()

## Case study: 2025's most widely congested moment

Everything above ranks borders by rent. This section asks a different question: when was the grid most *widely* constrained &ndash; the single moment with the most borders congested at once?

We scan every 15-minute timestamp across 2025, count how many borders are flagged `Congestion` at each one (`hourly_border_counts`), then take the max and list *every* timestamp that reaches it, rather than assuming the first one found is unique. That matters here: the max border count is a small integer, so ties across a whole year of 15-minute samples are expected, not a fluke.

In [ ]:
flows_2025 = fetch_flows_range("2025-01-01", "2026-01-01")
counts_2025 = hourly_border_counts(flows_2025, "Congestion")
total_borders = flows_2025["border"].nunique()

max_borders = counts_2025["n_borders"].max()
tied = counts_2025[counts_2025["n_borders"] == max_borders].sort_values("datetime")
print(
    f"Peak: {max_borders} of {total_borders} borders simultaneously congested, {len(tied)} timestamp(s) tied at that max:"
)
tied

The earliest of those tied peaks falls on 2025-01-28. The [VoltPath dashboard for that day](https://voltpath.ai/dashboard?viewMode=flows-prices&dateFrom=2025-01-28&dateTo=2025-01-29) shows it &ndash; drag the playback control to 11:15 UTC, since it defaults to midnight. The remaining tied timestamps fall on separate days in November 2025 and aren't charted here.

![VoltPath dashboard showing peak simultaneous congestion on 2025-01-28](media/congestion-peak-2025-01-28-dashboard.png)

The chart below plots that day's congested-border count against the average day-ahead price across all zones, with the peak moment marked. Both the day charted and the marker are derived from the scan above rather than hardcoded.

In [ ]:
peak_moment = tied["datetime"].iloc[0]
flows_peak_day = fetch_flows(
    peak_moment.strftime("%Y-%m-%d"),
    (peak_moment + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
    resolution="native",
)

fig = border_count_price_chart(
    flows_peak_day,
    "Congestion",
    f"Congested borders vs. average day-ahead price, {peak_moment:%Y-%m-%d} (UTC)",
    peak_vline=peak_moment.strftime("%Y-%m-%d %H:%M:%S"),
    peak_label="Peak congestion",
)
save_chart(fig, f"congestion-peak-{peak_moment:%Y-%m-%d}")
fig.show()

### Monthly leaderboard

Ranking by a single day isn't representative (see `flow_type_summary`'s docstring in `common.py`) &ndash; a full month is the unit worth ranking on. In terms of metrics, we came up with 4 options:

- `n_hours` and `volume_mwh` answer "how often" and "how much energy"
- `implied_value_eur` (congestion rent) answers "how much value the price gap is capturing on this border" &ndash; not economic cost or welfare lost, just the observed rent
- `avg_value_per_hour_eur` is an *intensity* metric (rare-but-severe vs. mild-but-frequent), not a welfare number &ndash; useful for spotting outlier hours, not for sizing total impact

All four panels come from `leaderboard_subplots` in `common.py`, and honour the `exclude_ukraine` flag set above.

Set `selected_month` below (e.g. `"2023-02"`) to pick a month; leave it `None` to default to the last full calendar month.

In [ ]:
selected_month = None  # e.g. "2023-02"; None = last full calendar month

month = (
    pd.Period(selected_month, freq="M")
    if selected_month
    else pd.Period(datetime.now(UTC).date(), freq="M") - 1
)
flows_month = fetch_flows_range(
    month.start_time.strftime("%Y-%m-%d"), (month + 1).start_time.strftime("%Y-%m-%d")
)
leaderboard_month = flow_type_summary(flows_month, "Congestion")
if exclude_ukraine:
    leaderboard_month = drop_ukraine(leaderboard_month)

fig = leaderboard_subplots(
    leaderboard_month, f"Top 10 borders by congestion metric, {month}"
)
save_chart(fig, f"monthly-leaderboard-{month}")
fig.show()

### Yearly leaderboard

Same four metrics, rolled up over a full calendar year instead of a month &ndash; useful for spotting borders that are a persistent, structural bottleneck rather than a one-month blip.

Set `selected_year` below (e.g. `2023`) to pick a year; leave it `None` to default to the last full calendar year. Fetching a full year re-fetches ~365 daily chunks the first time (cached afterwards, see `fetch_flows_range`'s docstring).

In [ ]:
selected_year = None  # e.g. 2023; None = last full calendar year

year = selected_year or (datetime.now(UTC).year - 1)
flows_year = fetch_flows_range(f"{year}-01-01", f"{year + 1}-01-01")
leaderboard_year = flow_type_summary(flows_year, "Congestion")
if exclude_ukraine:
    leaderboard_year = drop_ukraine(leaderboard_year)

fig = leaderboard_subplots(
    leaderboard_year, f"Top 10 borders by congestion metric, {year}"
)
save_chart(fig, f"yearly-leaderboard-{year}")
fig.show()

## Concentration check: how much of the rent do the top borders capture?

The leaderboards above show a steep drop-off, which is the usual basis for claiming a few borders account for most of the congestion rent. This measures that instead of asserting it.

The pies rank by `implied_value_eur` (congestion rent, EUR) specifically &ndash; not `n_hours` or `volume_mwh`, which would give a different ordering &ndash; and split each period's total rent into the top 10 borders by that metric vs. everyone else. Both inherit the `exclude_ukraine` setting from above.

In [ ]:
month_split = top_vs_rest(leaderboard_month)
year_split = top_vs_rest(leaderboard_year)

month_top_share = (
    1
    - month_split["implied_value_eur"].iloc[-1] / month_split["implied_value_eur"].sum()
)
year_top_share = (
    1 - year_split["implied_value_eur"].iloc[-1] / year_split["implied_value_eur"].sum()
)
print(
    f"Top 10 borders capture {month_top_share:.0%} of {month}'s rent "
    f"(of {len(leaderboard_month)} borders), "
    f"{year_top_share:.0%} of {year}'s (of {len(leaderboard_year)})."
)

fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "domain"}, {"type": "domain"}]],
    subplot_titles=[
        f"{month} (congestion rent, EUR)",
        f"{year} (congestion rent, EUR)",
    ],
)
fig.add_pie(
    labels=month_split["label"], values=month_split["implied_value_eur"], row=1, col=1
)
fig.add_pie(
    labels=year_split["label"], values=year_split["implied_value_eur"], row=1, col=2
)
fig.update_layout(
    title="Top 10 borders vs. everyone else, ranked by congestion rent (EUR, not hours or volume)",
    width=1400,
    height=600,
)
save_chart(fig, f"rent-concentration-{month}-vs-{year}")
fig.show()

## Congestion rent by region and country

Grouping borders by region gives a coarser view than the per-border leaderboard: is congestion rent concentrated within one part of Europe's grid, or spread evenly? Regions here are an approximate, geography-based grouping loosely modeled on ENTSO-E's [Capacity Calculation Regions](https://www.entsoe.eu/network_codes/ccr-regions/) (Nordic, Baltic, Core, Italy North, Greece-Italy, South-East Europe, South-West Europe), colored to match ENTSO-E's public CCR map. It's a simplification, not the real (overlapping) CCR membership: several TSOs legally sit in more than one CCR, which a one-region-per-zone mapping can't represent, so any border whose two sides fall in different regions is bucketed as "Cross-region" rather than forced into either side.

The treemap below drills region &rarr; country &rarr; border, sized and colored by `implied_value_eur` (congestion rent, EUR) for the same year as the yearly leaderboard above.

In [ ]:
treemap_data = leaderboard_year.copy()
treemap_data["region"] = treemap_data["border"].map(border_region)
treemap_data["country"] = treemap_data["border"].map(border_country)

fig = px.treemap(
    treemap_data,
    path=["region", "country", "border"],
    values="implied_value_eur",
    color="region",
    color_discrete_map=REGION_COLORS,
    title=f"Congestion rent by region and country, {year} (EUR)",
)
fig.update_layout(width=1400, height=800)
save_chart(fig, f"rent-by-region-country-{year}")
fig.show()

## Takeaways

- Congestion rent concentrates, but less sharply than the "a few borders dominate" framing suggests: the top 10 of ~75 borders take just over half the total in both periods measured above. Real concentration, not a handful of borders owning everything &ndash; quote the share printed above rather than a vaguer "most of it", since it shifts period to period.
- Which border is "worst" depends on the metric. Hours flagged, volume, total rent, and rent-per-hour produce visibly different top 10s in the leaderboards above, so any single ranking is a choice about the question being asked.
- The metric here is congestion rent, not deadweight loss &ndash; it's the value the price gap is capturing while the border stays constrained, not a claim about how much better off Europe would be if the bottleneck were removed. Rent is a defensible severity signal for ranking bottlenecks, not a benefit-case number for an interconnector investment; that needs the deadweight-loss triangle from the schematic near the top, which requires bid-curve data this API doesn't expose.
- Try narrowing `fetch_flows`/`fetch_flows_range` to a single border, or widening the window past 30 days/one month, to see how a specific bottleneck evolves.

---

Data: [ENTSO-E Transparency Platform](https://transparency.entsoe.eu), via the [Voltpath API](https://voltpath.ai).